In [1]:
from web_searching import web_search
from web_scraping import web_scrape
from llm_models import get_llm
from utilities import to_obj
import json 
import re
from prompts import (
    ASSISTANT_SELECTION_PROMPT_TEMPLATE,
    WEB_SEARCH_PROMPT_TEMPLATE,
    SUMMARY_PROMPT_TEMPLATE,
    RESEARCH_REPORT_PROMPT_TEMPLATE
)

NUM_SEARCH_QUERIES = 5
NUM_SEARCH_RESULTS_PER_QUERY = 3
RESULT_TEXT_MAX_CHARACTERS = 10000


In [2]:
question = 'What can I see and do in the Spanish town of Astorga?'

llm = get_llm()
assistant_selection_prompt = ASSISTANT_SELECTION_PROMPT_TEMPLATE.format(user_question=question)
assistant_instructions = llm.invoke(assistant_selection_prompt)

In [3]:
assistant_instructions

AIMessage(content='Based on the user\'s question "What can I see and do in the Spanish town of Astorga?", I would assign the following assistant:\n\n{\n    "assistant_type": "Tour guide assistant",\n    "assistant_instructions": "You are a world-travelled AI tour guide assistant. Your main purpose is to draft engaging, insightful, unbiased, and well-structured travel reports on given locations, including history, attractions, and cultural insights.",\n}', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-08-09T21:28:39.2256328Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1281519700, 'load_duration': 186656900, 'prompt_eval_count': 434, 'prompt_eval_duration': 15903000, 'eval_count': 91, 'eval_duration': 1037425000, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019fe86d-7741-7372-be10-cf6a155d971d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 434, 'output_tokens': 91, 't

In [4]:
print(assistant_instructions.content)

Based on the user's question "What can I see and do in the Spanish town of Astorga?", I would assign the following assistant:

{
    "assistant_type": "Tour guide assistant",
    "assistant_instructions": "You are a world-travelled AI tour guide assistant. Your main purpose is to draft engaging, insightful, unbiased, and well-structured travel reports on given locations, including history, attractions, and cultural insights.",
}


In [5]:
assistant_instructions_dict = to_obj(assistant_instructions.content)

In [6]:
assistant_instructions_dict

{'assistant_type': 'Tour guide assistant',
 'assistant_instructions': 'You are a world-travelled AI tour guide assistant. Your main purpose is to draft engaging, insightful, unbiased, and well-structured travel reports on given locations, including history, attractions, and cultural insights.'}

In [7]:
web_search_prompt = WEB_SEARCH_PROMPT_TEMPLATE.format(
    assistant_instructions=assistant_instructions_dict[
    'assistant_instructions'],
    num_search_queries=NUM_SEARCH_QUERIES,
    user_question=question)


In [8]:
print(web_search_prompt)


You are a world-travelled AI tour guide assistant. Your main purpose is to draft engaging, insightful, unbiased, and well-structured travel reports on given locations, including history, attractions, and cultural insights.

Write 5 web search queries to gather as much information as possible 
on the following question: What can I see and do in the Spanish town of Astorga?. Your objective is to write a report based on the information you find.
You must respond with a list of queries such as query1, query2, query3 in the following format: 
[
    {"search_query": "query1", "user_question": "What can I see and do in the Spanish town of Astorga?" },
    {"search_query": "query2", "user_question": "What can I see and do in the Spanish town of Astorga?" },
    {"search_query": "query3", "user_question": "What can I see and do in the Spanish town of Astorga?" }
]



In [9]:
web_search_queries = llm.invoke(web_search_prompt)


In [10]:
web_search_queries

AIMessage(content='Here are 5 web search queries to gather information on what to see and do in the Spanish town of Astorga:\n\n[\n    {"search_query": "Astorga Spain tourist attractions", "user_question": "What can I see and do in the Spanish town of Astorga?" },\n    {"search_query": "Things to do in Astorga province León", "user_question": "What can I see and do in the Spanish town of Astorga?" },\n    {"search_query": "Astorga Spain historical landmarks", "user_question": "What can I see and do in the Spanish town of Astorga?" },\n    {"search_query": "Best places to visit in Astorga León", "user_question": "What can I see and do in the Spanish town of Astorga?" },\n    {"search_query": "Astorga Spain cultural activities", "user_question": "What can I see and do in the Spanish town of Astorga?" }\n]', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-08-09T21:28:41.8812964Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2577937800, 'load

In [11]:
web_search_queries_list = to_obj(
    web_search_queries.content.replace('\n', ''))

In [14]:
print(web_search_queries_list)

[{'search_query': 'Astorga Spain tourist attractions', 'user_question': 'What can I see and do in the Spanish town of Astorga?'}, {'search_query': 'Things to do in Astorga province León', 'user_question': 'What can I see and do in the Spanish town of Astorga?'}, {'search_query': 'Astorga Spain historical landmarks', 'user_question': 'What can I see and do in the Spanish town of Astorga?'}, {'search_query': 'Best places to visit in Astorga León', 'user_question': 'What can I see and do in the Spanish town of Astorga?'}, {'search_query': 'Astorga Spain cultural activities', 'user_question': 'What can I see and do in the Spanish town of Astorga?'}]


In [19]:
searches_and_result_urls = [{
    'result_urls': web_search(
   web_query=wq['search_query'], 
   num_results=NUM_SEARCH_RESULTS_PER_QUERY), 
    'search_query': wq['search_query']} 
   for wq in web_search_queries_list]

[{'search_query': 'Astorga Spain tourist attractions',
  'user_question': 'What can I see and do in the Spanish town of Astorga?'},
 {'search_query': 'Things to do in Astorga province León',
  'user_question': 'What can I see and do in the Spanish town of Astorga?'},
 {'search_query': 'Astorga Spain historical landmarks',
  'user_question': 'What can I see and do in the Spanish town of Astorga?'},
 {'search_query': 'Best places to visit in Astorga León',
  'user_question': 'What can I see and do in the Spanish town of Astorga?'},
 {'search_query': 'Astorga Spain cultural activities',
  'user_question': 'What can I see and do in the Spanish town of Astorga?'}]

In [16]:
searches_and_result_urls

[{'result_urls': ['https://en.wikipedia.org/wiki/Astorga,_Spain',
   'https://www.komoot.com/guide/1771715/attractions-around-astorga',
   'https://www.terranostrum.es/turismo/un-paseo-por-astorga'],
  'search_query': 'Astorga Spain tourist attractions'},
 {'result_urls': ['https://www.terranostrum.es/turismo/un-paseo-por-astorga',
   'https://www.quieroviajarsola.com/espana/castilla-y-leon/que-ver-en-astorga/',
   'https://www.komoot.com/guide/1568259/attractions-around-astorga'],
  'search_query': 'Things to do in Astorga province León'},
 {'result_urls': ['https://en.wikipedia.org/wiki/Astorga,_Spain',
   'https://www.bing.com/aclick?ld=e8H3Fj6jcYb0is2M1iL1bhyTVUCUyNUqoIAfHr1IRHpd878Z_r3KuxqVnW7bXTLg_r0e96s07ZqI1cYnOqqlaH3dCfc_MvfzH6BznK4oywy1-7mivarAzHsO4xEYlxu3MMrJz_zlNcxnfRvEUbmpH1rq9blPXbnB1IJCvSrom8ac-2Wqeqki30IhMVD9NdwncykQioPw&u=aHR0cHMlM2ElMmYlMmZ3d3cuZ2V0eW91cmd1aWRlLmNvbSUyZi1sMTgzNTM2JTJmJTNmY21wJTNkYmluZyUyNmNtcCUzZGJpbmclMjZhZF9pZCUzZDc4Njg0MDIwNTQ2MDY0JTI2YWRncm91cF9pZ

In [21]:
search_query_and_result_url_list = []
for qr in searches_and_result_urls:
    search_query_and_result_url_list.extend([{
        'search_query': qr['search_query'], 
        'result_url': r} 
           for r in qr['result_urls']])

In [22]:
search_query_and_result_url_list

[{'search_query': 'Astorga Spain tourist attractions',
  'result_url': 'https://www.bing.com/aclick?ld=e8Xh68Bz-Y81_HmdkJZ_7ykjVUCUwTiaNu_wYFdbJSpUgsgoXS4qwI6PokIKLg6gYAzqpFkoEoVj5MID2sI4VvALC7n4wlgQsnWOCLRYPdQXb5ohigF9d9YXFpcIostBLQrny8YzL1LIPNolTn0xOPW0V3z6MrpZIJHf7PuuGe1vmPy90_WOjn5YonlGt_3tRxuKoVNw&u=aHR0cHMlM2ElMmYlMmZ3d3cuZ2V0eW91cmd1aWRlLmNvbSUyZi1sMTgzNTM2JTJmJTNmY21wJTNkYmluZyUyNmNtcCUzZGJpbmclMjZhZF9pZCUzZDc4Njg0MDIwNTQ2MDY0JTI2YWRncm91cF9pZCUzZDEyNTg5NDIxNzQ5OTQ4MzElMjZiaWRfbWF0Y2hfdHlwZSUzZGJwJTI2Y2FtcGFpZ25faWQlM2Q3MTA5MzM4NjklMjZkZXZpY2UlM2RjJTI2ZmVlZF9pdGVtX2lkJTNkJTI2a2V5d29yZCUzZGFzdG9yZ2ElMjUyMHNwYWluJTI2bG9jX2ludGVyZXN0X21zJTNkMzE5OSUyNmxvY19waHlzaWNhbF9tcyUzZDE2NDY5OCUyNm1hdGNoX3R5cGUlM2RwJTI2bXNjbGtpZCUzZGJhMjNlN2M3MjMzNzE0NzIwY2EwOGRiZmQyYzcwNTBhJTI2bmV0d29yayUzZG8lMjZwYXJ0bmVyX2lkJTNkQ0Q5NTElMjZ0YXJnZXRfaWQlM2Rrd2QtNzg2ODQyNzQ1OTc4NjMlM2Fsb2MtMTcwJTI2dXRtX2FkZ3JvdXAlM2RsYyUyNTNEMTgzNTM2JTI1M0Fhc3RvcmdhJTI1N0NmbiUyNTNEZjMlMjU3Q2NpJTI1M0Q5MzclMjUzQXRoaW5ncyUyNTIwdG

In [25]:
result_text_list = [{
    'result_text': web_scrape(
    url=re['result_url'])[:RESULT_TEXT_MAX_CHARACTERS],
    'result_url': re['result_url'],
    'search_query': re['search_query']}
    for re in search_query_and_result_url_list]

In [26]:
result_text_list

[{'result_text': 'Failed to retrieve the webpage: Status code 403',
  'result_url': 'https://www.bing.com/aclick?ld=e8Xh68Bz-Y81_HmdkJZ_7ykjVUCUwTiaNu_wYFdbJSpUgsgoXS4qwI6PokIKLg6gYAzqpFkoEoVj5MID2sI4VvALC7n4wlgQsnWOCLRYPdQXb5ohigF9d9YXFpcIostBLQrny8YzL1LIPNolTn0xOPW0V3z6MrpZIJHf7PuuGe1vmPy90_WOjn5YonlGt_3tRxuKoVNw&u=aHR0cHMlM2ElMmYlMmZ3d3cuZ2V0eW91cmd1aWRlLmNvbSUyZi1sMTgzNTM2JTJmJTNmY21wJTNkYmluZyUyNmNtcCUzZGJpbmclMjZhZF9pZCUzZDc4Njg0MDIwNTQ2MDY0JTI2YWRncm91cF9pZCUzZDEyNTg5NDIxNzQ5OTQ4MzElMjZiaWRfbWF0Y2hfdHlwZSUzZGJwJTI2Y2FtcGFpZ25faWQlM2Q3MTA5MzM4NjklMjZkZXZpY2UlM2RjJTI2ZmVlZF9pdGVtX2lkJTNkJTI2a2V5d29yZCUzZGFzdG9yZ2ElMjUyMHNwYWluJTI2bG9jX2ludGVyZXN0X21zJTNkMzE5OSUyNmxvY19waHlzaWNhbF9tcyUzZDE2NDY5OCUyNm1hdGNoX3R5cGUlM2RwJTI2bXNjbGtpZCUzZGJhMjNlN2M3MjMzNzE0NzIwY2EwOGRiZmQyYzcwNTBhJTI2bmV0d29yayUzZG8lMjZwYXJ0bmVyX2lkJTNkQ0Q5NTElMjZ0YXJnZXRfaWQlM2Rrd2QtNzg2ODQyNzQ1OTc4NjMlM2Fsb2MtMTcwJTI2dXRtX2FkZ3JvdXAlM2RsYyUyNTNEMTgzNTM2JTI1M0Fhc3RvcmdhJTI1N0NmbiUyNTNEZjMlMjU3Q2NpJTI1M0Q5MzclMjUzQXRoa

In [36]:
result_text_summary_list = []
for rt in result_text_list: 
    summary_prompt = SUMMARY_PROMPT_TEMPLATE.format(
    search_result_text=rt['result_text'], 
    search_query=rt['search_query'])

    text_summary = llm.invoke(summary_prompt)

    result_text_summary_list.append({
    'text_summary': text_summary.content,
    'result_url': rt['result_url'],
    'search_query': rt['search_query']})

In [37]:
result_text_summary_list

[{'text_summary': 'I\'ll be happy to help.\n\nSince there\'s no relation between the two pieces of text, I will provide a summary of the given text:\n\nThe text indicates that an attempt was made to retrieve a webpage, but it failed with a status code of 403. This is known as a "Forbidden" error, which means the server refused to access the requested resource due to security restrictions or permissions issues.\n\nNo factual information, numbers, stats, or context related to Astorga Spain tourist attractions can be found in this text.',
  'result_url': 'https://www.bing.com/aclick?ld=e8Xh68Bz-Y81_HmdkJZ_7ykjVUCUwTiaNu_wYFdbJSpUgsgoXS4qwI6PokIKLg6gYAzqpFkoEoVj5MID2sI4VvALC7n4wlgQsnWOCLRYPdQXb5ohigF9d9YXFpcIostBLQrny8YzL1LIPNolTn0xOPW0V3z6MrpZIJHf7PuuGe1vmPy90_WOjn5YonlGt_3tRxuKoVNw&u=aHR0cHMlM2ElMmYlMmZ3d3cuZ2V0eW91cmd1aWRlLmNvbSUyZi1sMTgzNTM2JTJmJTNmY21wJTNkYmluZyUyNmNtcCUzZGJpbmclMjZhZF9pZCUzZDc4Njg0MDIwNTQ2MDY0JTI2YWRncm91cF9pZCUzZDEyNTg5NDIxNzQ5OTQ4MzElMjZiaWRfbWF0Y2hfdHlwZSUzZGJwJTI

In [38]:
stringified_summary_list = [
    f'Source URL: {sr["result_url"]}\nSummary: {sr["text_summary"]}' 
        for sr in result_text_summary_list]

In [39]:
print(stringified_summary_list[0])

Source URL: https://www.bing.com/aclick?ld=e8Xh68Bz-Y81_HmdkJZ_7ykjVUCUwTiaNu_wYFdbJSpUgsgoXS4qwI6PokIKLg6gYAzqpFkoEoVj5MID2sI4VvALC7n4wlgQsnWOCLRYPdQXb5ohigF9d9YXFpcIostBLQrny8YzL1LIPNolTn0xOPW0V3z6MrpZIJHf7PuuGe1vmPy90_WOjn5YonlGt_3tRxuKoVNw&u=aHR0cHMlM2ElMmYlMmZ3d3cuZ2V0eW91cmd1aWRlLmNvbSUyZi1sMTgzNTM2JTJmJTNmY21wJTNkYmluZyUyNmNtcCUzZGJpbmclMjZhZF9pZCUzZDc4Njg0MDIwNTQ2MDY0JTI2YWRncm91cF9pZCUzZDEyNTg5NDIxNzQ5OTQ4MzElMjZiaWRfbWF0Y2hfdHlwZSUzZGJwJTI2Y2FtcGFpZ25faWQlM2Q3MTA5MzM4NjklMjZkZXZpY2UlM2RjJTI2ZmVlZF9pdGVtX2lkJTNkJTI2a2V5d29yZCUzZGFzdG9yZ2ElMjUyMHNwYWluJTI2bG9jX2ludGVyZXN0X21zJTNkMzE5OSUyNmxvY19waHlzaWNhbF9tcyUzZDE2NDY5OCUyNm1hdGNoX3R5cGUlM2RwJTI2bXNjbGtpZCUzZGJhMjNlN2M3MjMzNzE0NzIwY2EwOGRiZmQyYzcwNTBhJTI2bmV0d29yayUzZG8lMjZwYXJ0bmVyX2lkJTNkQ0Q5NTElMjZ0YXJnZXRfaWQlM2Rrd2QtNzg2ODQyNzQ1OTc4NjMlM2Fsb2MtMTcwJTI2dXRtX2FkZ3JvdXAlM2RsYyUyNTNEMTgzNTM2JTI1M0Fhc3RvcmdhJTI1N0NmbiUyNTNEZjMlMjU3Q2NpJTI1M0Q5MzclMjUzQXRoaW5ncyUyNTIwdG8lMjUyMGRvJTI2dXRtX2NhbXBhaWduJTNkZGMlMjUzRDIxJTI1M0FlcyUyNT

In [40]:
appended_result_summaries = '\n'.join(stringified_summary_list)

In [41]:
appended_result_summaries

'Source URL: https://www.bing.com/aclick?ld=e8Xh68Bz-Y81_HmdkJZ_7ykjVUCUwTiaNu_wYFdbJSpUgsgoXS4qwI6PokIKLg6gYAzqpFkoEoVj5MID2sI4VvALC7n4wlgQsnWOCLRYPdQXb5ohigF9d9YXFpcIostBLQrny8YzL1LIPNolTn0xOPW0V3z6MrpZIJHf7PuuGe1vmPy90_WOjn5YonlGt_3tRxuKoVNw&u=aHR0cHMlM2ElMmYlMmZ3d3cuZ2V0eW91cmd1aWRlLmNvbSUyZi1sMTgzNTM2JTJmJTNmY21wJTNkYmluZyUyNmNtcCUzZGJpbmclMjZhZF9pZCUzZDc4Njg0MDIwNTQ2MDY0JTI2YWRncm91cF9pZCUzZDEyNTg5NDIxNzQ5OTQ4MzElMjZiaWRfbWF0Y2hfdHlwZSUzZGJwJTI2Y2FtcGFpZ25faWQlM2Q3MTA5MzM4NjklMjZkZXZpY2UlM2RjJTI2ZmVlZF9pdGVtX2lkJTNkJTI2a2V5d29yZCUzZGFzdG9yZ2ElMjUyMHNwYWluJTI2bG9jX2ludGVyZXN0X21zJTNkMzE5OSUyNmxvY19waHlzaWNhbF9tcyUzZDE2NDY5OCUyNm1hdGNoX3R5cGUlM2RwJTI2bXNjbGtpZCUzZGJhMjNlN2M3MjMzNzE0NzIwY2EwOGRiZmQyYzcwNTBhJTI2bmV0d29yayUzZG8lMjZwYXJ0bmVyX2lkJTNkQ0Q5NTElMjZ0YXJnZXRfaWQlM2Rrd2QtNzg2ODQyNzQ1OTc4NjMlM2Fsb2MtMTcwJTI2dXRtX2FkZ3JvdXAlM2RsYyUyNTNEMTgzNTM2JTI1M0Fhc3RvcmdhJTI1N0NmbiUyNTNEZjMlMjU3Q2NpJTI1M0Q5MzclMjUzQXRoaW5ncyUyNTIwdG8lMjUyMGRvJTI2dXRtX2NhbXBhaWduJTNkZGMlMjUzRDIxJTI1M0FlcyUyN

In [43]:
research_report_prompt = RESEARCH_REPORT_PROMPT_TEMPLATE.format(
    research_summary=appended_result_summaries,
    user_question=question
)
research_report = llm.invoke(research_report_prompt)

print(f'strigified_summary_list={stringified_summary_list}')
print(f'merged_result_summaries={appended_result_summaries}')
print(f'research_report={research_report}')

strigified_summary_list=['Source URL: https://www.bing.com/aclick?ld=e8Xh68Bz-Y81_HmdkJZ_7ykjVUCUwTiaNu_wYFdbJSpUgsgoXS4qwI6PokIKLg6gYAzqpFkoEoVj5MID2sI4VvALC7n4wlgQsnWOCLRYPdQXb5ohigF9d9YXFpcIostBLQrny8YzL1LIPNolTn0xOPW0V3z6MrpZIJHf7PuuGe1vmPy90_WOjn5YonlGt_3tRxuKoVNw&u=aHR0cHMlM2ElMmYlMmZ3d3cuZ2V0eW91cmd1aWRlLmNvbSUyZi1sMTgzNTM2JTJmJTNmY21wJTNkYmluZyUyNmNtcCUzZGJpbmclMjZhZF9pZCUzZDc4Njg0MDIwNTQ2MDY0JTI2YWRncm91cF9pZCUzZDEyNTg5NDIxNzQ5OTQ4MzElMjZiaWRfbWF0Y2hfdHlwZSUzZGJwJTI2Y2FtcGFpZ25faWQlM2Q3MTA5MzM4NjklMjZkZXZpY2UlM2RjJTI2ZmVlZF9pdGVtX2lkJTNkJTI2a2V5d29yZCUzZGFzdG9yZ2ElMjUyMHNwYWluJTI2bG9jX2ludGVyZXN0X21zJTNkMzE5OSUyNmxvY19waHlzaWNhbF9tcyUzZDE2NDY5OCUyNm1hdGNoX3R5cGUlM2RwJTI2bXNjbGtpZCUzZGJhMjNlN2M3MjMzNzE0NzIwY2EwOGRiZmQyYzcwNTBhJTI2bmV0d29yayUzZG8lMjZwYXJ0bmVyX2lkJTNkQ0Q5NTElMjZ0YXJnZXRfaWQlM2Rrd2QtNzg2ODQyNzQ1OTc4NjMlM2Fsb2MtMTcwJTI2dXRtX2FkZ3JvdXAlM2RsYyUyNTNEMTgzNTM2JTI1M0Fhc3RvcmdhJTI1N0NmbiUyNTNEZjMlMjU3Q2NpJTI1M0Q5MzclMjUzQXRoaW5ncyUyNTIwdG8lMjUyMGRvJTI2dXRtX2NhbXBhaWduJTNk